# DPI Probe Gate: all languages for dhee-pravega (Kaggle)

Translated scenarios are UNREVIEWED LLM drafts (human_reviewed=False). English data comes from kernel `rajtaware10/safl-dpi-english`. Collect, Sarvam-M judge and probes for hi hi_en mr mr_en ta ta_en. Logs in `/kaggle/working/logs/`.

In [ ]:
# The GitHub repo is private, so the code comes in as a private Kaggle dataset
# (dataset_sources in kernel-metadata.json, or Add Input in the UI).
import glob, os, shutil, zipfile
!nvidia-smi -L
W = "/kaggle/working/safl-hackathon"
hits = glob.glob("/kaggle/input/**/collect.py", recursive=True)
if hits:
    shutil.copytree(os.path.dirname(hits[0]), W, dirs_exist_ok=True)
else:
    zips = glob.glob("/kaggle/input/**/code.zip", recursive=True)
    assert zips, "code dataset not attached: nothing under /kaggle/input"
    zipfile.ZipFile(zips[0]).extractall(W)
%cd {W}
!ls -R | head -40

In [ ]:
# Gemma-4 (dhee-pravega) needs a recent transformers; 5.7.0 is the version checked locally.
!pip install -q "transformers==5.7.0" accelerate
import transformers, torch
print(transformers.__version__, torch.__version__, torch.cuda.device_count(), "GPUs")

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN set from Kaggle secrets")
except Exception as e:
    print("No HF_TOKEN secret; public downloads only:", type(e).__name__)

In [ ]:
!mkdir -p /kaggle/working/logs && df -h / /kaggle/working | tail -2

In [ ]:
# English acts/meta/judge scores from the English run
import glob, zipfile
z = glob.glob('/kaggle/input/**/english_outputs.zip', recursive=True)
assert z, 'attach kernel output rajtaware10/safl-dpi-english'
zipfile.ZipFile(z[0]).extractall('.')
!ls data/dhee-pravega

In [ ]:
!python collect.py --model dhee-pravega --langs hi hi_en mr mr_en ta ta_en --batch-size 8 2>&1 | tee /kaggle/working/logs/collect_dhee_all.log

In [ ]:
# Free disk for Sarvam-M (~48 GB download).
!rm -rf ~/.cache/huggingface/hub/models--Qwen--Qwen3-8B ~/.cache/huggingface/hub/models--dheeyantra--dhee-pravega; df -h / | tail -1

In [ ]:
!pip install -q bitsandbytes
!python judge.py --agent dhee-pravega --langs hi hi_en mr mr_en ta ta_en --load-4bit 2>&1 | tee /kaggle/working/logs/judge_dhee_all.log

In [ ]:
!pip install -q scikit-learn matplotlib
!python probes.py --model dhee-pravega --langs en hi hi_en mr mr_en ta ta_en 2>&1 | tee /kaggle/working/logs/probes_dhee_all.log

In [ ]:
!zip -q -r /kaggle/working/all_outputs_dhee.zip data/dhee-pravega results && ls -la /kaggle/working/all_outputs_dhee.zip